# UQRoute-TC — Llama 3B corrected pilot

Run the six code cells in order. This notebook uses the fixed validator at commit `8f919e3`. If the five earlier responses are in Drive, section 3 validates and repairs only the evidence, saves the originals, and skips model startup. If no responses exist, sections 4–5 run the five-case pilot on an L4. The last section scores and verifies the result.

Open this notebook in Colab and connect a runtime. You may keep the current L4 runtime; the saved responses are on Drive. Stop at any error and share that cell's final lines.


## 1. Connect Drive and restore pinned source


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')
assert Path('/content/drive/MyDrive/UQRoute-TC').is_dir(), 'UQRoute-TC Drive folder missing'

benchmark = Path('/content/robustbench-tc-release')
if not benchmark.exists():
    subprocess.run(['git', 'clone',
        'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision

project = Path('/content/uqroute-tc-llama-fixed')
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', 'feat/task1-near-reproduction-split',
        '--single-branch', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
pilot_revision = '8f919e3b8a977b35c312ed748cb796943b48612e'
subprocess.run(['git', '-C', str(project), 'fetch', 'origin',
    'feat/task1-near-reproduction-split'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', pilot_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == pilot_revision
print('Drive, benchmark, and corrected pilot source ready:', pilot_revision)


## 2. Install the light pilot dependencies


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'openai>=1.50,<3', 'tqdm>=4.66,<5', '-e', str(project)], check=True)
sys.path.insert(0, str(project / 'src'))
from uqroute_tc.inference.pilot import _atomic_json, _record_path, token_evidence
print('Corrected validator available in this notebook kernel')


## 3. Validate and recover any saved Llama responses


In [ ]:
import json
import os
import shutil

model_id = 'meta-llama/Llama-3.2-3B-Instruct'
model_revision = '0cb88a4f764b7a12671c53f0838cd831a0843b95'
output_dir = Path('/content/drive/MyDrive/UQRoute-TC/pilots/llama32-3b-dev-five-v1')
manifest_path = output_dir / 'manifest.json'
need_model = not manifest_path.exists()

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert manifest['model_id'] == model_id and manifest['model_revision'] == model_revision
    assert manifest['benchmark_revision'] == benchmark_revision
    assert manifest['code_revision'] in {'69805b9341306692ae4a412a0a9a8939cd156504', pilot_revision}
    case_ids = manifest['case_ids']
    assert len(case_ids) == 5 and len(set(case_ids)) == 5

    repairs = []
    for case_id in case_ids:
        path = _record_path(output_dir, case_id)
        if not path.exists():
            need_model = True
            continue
        record = json.loads(path.read_text())
        assert record['id'] == case_id
        uq = record['uqroute']
        if uq['status'] == 'complete':
            assert uq['evidence']['valid']
            continue
        if uq['status'] == 'invalid_evidence':
            assert uq['evidence']['reason'] == 'chosen-token bytes do not match visible UTF-8 output'
            assert uq['finish_reason'] == 'stop'
            choice = uq['response']['choices'][0]
            assert choice['logprobs']['content'][-1]['token'] == '<|eot_id|>'
            evidence = token_evidence(choice)
            assert evidence['valid'] and evidence['excluded_trailing_token'] == '<|eot_id|>'
            uq['evidence_repair'] = {
                'source': 'saved response', 'original_evidence': uq['evidence'],
                'validator_commit': pilot_revision,
                'rule': 'exclude verified trailing <|eot_id|> after exact byte alignment',
            }
            uq['evidence'] = evidence
            uq['status'] = 'complete'
            uq['error'] = None
            repairs.append((path, record))
        else:
            need_model = True

    if need_model and manifest['code_revision'] != pilot_revision:
        raise RuntimeError('Saved run has missing/incomplete cases under old code; retain it and use a new output directory for a new run')

    if repairs:
        backup = output_dir / 'records_before_eot_repair'
        backup.mkdir(exist_ok=True)
        for path, record in repairs:
            shutil.copy2(path, backup / path.name)
            _atomic_json(path, record)
        _atomic_json(output_dir / 'eot_repair_receipt.json', {
            'original_code_revision': manifest['code_revision'],
            'validator_commit': pilot_revision,
            'case_ids': [record['id'] for _, record in repairs],
            'backup_directory': backup.name,
        })

    if not need_model:
        records = [json.loads(_record_path(output_dir, case_id).read_text()) for case_id in case_ids]
        assert all(record['uqroute']['status'] == 'complete' and
                   record['uqroute']['evidence']['valid'] for record in records)
        predictions = output_dir / 'pilot.predictions.jsonl'
        temporary = predictions.with_name(predictions.name + '.tmp')
        with temporary.open('w', encoding='utf-8') as stream:
            for record in records:
                stream.write(json.dumps(record, ensure_ascii=False) + '\n')
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, predictions)
        print('Saved evidence verified:', len(records), 'cases; new inference is unnecessary')
        print('Original records backed up:', len(repairs))

if need_model:
    print('Some cases have no usable saved response; the next section will start Llama')


## 4. Start Llama only if inference is needed


In [ ]:
if need_model:
    import importlib.metadata
    import time
    import urllib.request
    from google.colab import userdata
    from huggingface_hub import hf_hub_download

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.30.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
    import torch
    assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), 'Select an L4 GPU runtime'
    hf_token = userdata.get('HF_TOKEN')
    if not hf_token:
        raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access')
    hf_hub_download(model_id, 'config.json', revision=model_revision, token=hf_token)
    os.environ['HF_TOKEN'] = hf_token

    endpoint = 'http://127.0.0.1:8000/v1'
    server_log = Path('/content/uqroute_llama_fixed_server.log')
    def served_models():
        try:
            with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
                return [m['id'] for m in json.load(response)['data']]
        except Exception:
            return None

    active = served_models()
    if active is not None and model_id not in active:
        raise RuntimeError(f'Port 8000 serves a different model: {active}')
    if model_id not in (active or []):
        existing = subprocess.check_output(['ps', '-eo', 'args='], text=True)
        server = None
        if f'vllm serve {model_id}' not in existing:
            with server_log.open('w') as log:
                server = subprocess.Popen([
                    'vllm', 'serve', model_id, '--revision', model_revision,
                    '--tokenizer-revision', model_revision, '--dtype', 'bfloat16',
                    '--max-model-len', '8192', '--gpu-memory-utilization', '0.88',
                    '--port', '8000',
                ], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            print('Started server PID:', server.pid)
        for _ in range(120):
            if model_id in (served_models() or []):
                break
            if server is not None and server.poll() is not None:
                print('\n'.join(server_log.read_text(errors='replace').splitlines()[-40:]))
                raise RuntimeError('Llama server exited during startup')
            time.sleep(5)
        else:
            if server_log.exists():
                print('\n'.join(server_log.read_text(errors='replace').splitlines()[-40:]))
            raise TimeoutError('Llama server did not become ready')
    print('Server ready:', model_id)
else:
    print('Server skipped: five saved cases already have valid evidence')


## 5. Run or resume the five-case pilot


In [ ]:
if need_model:
    import importlib.metadata
    command = [
        sys.executable, '-m', 'uqroute_tc.inference.pilot',
        '--benchmark-root', str(benchmark),
        '--split-manifest', str(project / 'docs/splits/group_split_v1.json'),
        '--output-dir', str(output_dir), '--model-id', model_id,
        '--model-revision', model_revision,
        '--tokenizer-revision', model_revision,
        '--endpoint', 'http://127.0.0.1:8000/v1',
        '--engine-version', importlib.metadata.version('vllm'),
        '--dtype', 'bfloat16', '--max-model-len', '8192',
        '--temperature', '0.001', '--max-tokens', '1024',
        '--top-logprobs', '5', '--limit', '5',
    ]
    subprocess.run(command, check=True)
else:
    print('Five saved responses verified and exported; no new model requests')


## 6. Score and verify


In [ ]:
subprocess.run([
    sys.executable,
    str(benchmark / 'leaderboard/scoring/eval_results.py'),
    '--predictions', str(output_dir),
    '--output', str(output_dir / 'scores.json'),
], check=True)

rows = [json.loads(line) for line in
        (output_dir / 'pilot.predictions.jsonl').read_text().splitlines()]
assert len(rows) == 5 and len({row['id'] for row in rows}) == 5
assert all(row['uqroute']['status'] == 'complete' and
           row['uqroute']['evidence']['valid'] for row in rows)
print('Verified: five unique scored-ready Llama records with valid token evidence')
print('Saved in:', output_dir)
